# 💾 FarmIntel Stage 1 — Save Trained Artifacts

**Purpose:** Save all trained artifacts required for production inference.  
**Selected Model:** XGBoost (Top-3 Acc ≈ Gradient Boosting; training time 33× faster: 105s vs 3439s)  

### Artifacts saved
| File | Description |
|------|-------------|
| `crop_category_xgboost.pkl` | Trained XGBoost classifier |
| `ordinal_encoder.pkl` | Fitted `OrdinalEncoder` for input features |
| `label_encoder.pkl` | Fitted `LabelEncoder` for crop categories |
| `feature_columns.json` | Ordered feature list used during training |
| `crop_categories.json` | Ordered class names |
| `metadata.json` | Full model metadata for deployment |

---
## 1. Imports

In [5]:
import json
import sys
import time
import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from xgboost              import XGBClassifier
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics       import accuracy_score, f1_score

print(f'Python     : {sys.version.split()[0]}')
print(f'sklearn    : {sklearn.__version__}')
print(f'xgboost    : {xgb.__version__}')
print(f'joblib     : {joblib.__version__}')

Python     : 3.10.10
sklearn    : 1.7.2
xgboost    : 3.2.0
joblib     : 1.5.3


---
## 2. Configuration

In [6]:
# ── Model choice ─────────────────────────────────────────────────────
# XGBoost selected over Gradient Boosting:
#   Top-3 Acc : 0.8593  vs  0.8594  (Δ = 0.0001, negligible)
#   Train time:  105s   vs  3439s   (33× faster)
SELECTED_MODEL_NAME = 'XGBoost'

# ── Paths ─────────────────────────────────────────────────────────────
DATA_DIR   = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# ── Feature / target definitions ─────────────────────────────────────
CAT_FEATURES = ['State', 'District', 'Season']
NUM_FEATURES = ['Year']
ALL_FEATURES = CAT_FEATURES + NUM_FEATURES
TARGET       = 'Crop_Category'
TOP_K        = 3
RANDOM_STATE = 42

# ── XGBoost hyperparameters (identical to model selection) ───────────
XGBOOST_PARAMS = dict(
    n_estimators      = 300,
    learning_rate     = 0.05,
    max_depth         = 7,
    subsample         = 0.8,
    colsample_bytree  = 0.8,
    eval_metric       = 'mlogloss',
    use_label_encoder = False,
    n_jobs            = -1,
    random_state      = RANDOM_STATE,
    verbosity         = 0,
)

print(f'Data dir   : {DATA_DIR.resolve()}')
print(f'Models dir : {MODELS_DIR.resolve()}')
print(f'Features   : {ALL_FEATURES}')
print(f'Target     : {TARGET}')

Data dir   : D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\data\processed
Models dir : D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models
Features   : ['State', 'District', 'Season', 'Year']
Target     : Crop_Category


---
## 3. Helper Functions

In [8]:
def top_k_accuracy(y_true: np.ndarray, proba: np.ndarray, k: int = 3) -> float:
    """
    Vectorised Top-K accuracy from predict_proba() output.
    A prediction is correct if the true label appears in the K
    highest-probability classes.
    """
    top_k_idx = np.argsort(proba, axis=1)[:, -k:]
    correct   = np.any(top_k_idx == y_true.reshape(-1, 1), axis=1)
    return float(correct.mean())


def evaluate_on_test(
    model,
    X_te: np.ndarray,
    y_te: np.ndarray,
    k: int = TOP_K,
) -> dict:
    """
    Compute inference metrics on the held-out test set.
    Returns a plain dict of metric_name → float.
    """
    y_pred = model.predict(X_te)
    proba  = model.predict_proba(X_te)

    return {
        'test_accuracy'   : round(accuracy_score(y_te, y_pred), 6),
        f'top{k}_accuracy': round(top_k_accuracy(y_te, proba, k=k), 6),
        'weighted_f1'     : round(f1_score(y_te, y_pred, average='weighted', zero_division=0), 6),
        'macro_f1'        : round(f1_score(y_te, y_pred, average='macro',    zero_division=0), 6),
    }


def save_artifact(obj, path: Path, use_json: bool = False) -> None:
    """
    Save a single artifact to disk.

    Parameters
    ----------
    obj      : the object to save
    path     : pathlib.Path destination
    use_json : if True, dump as JSON; otherwise use joblib
    """
    if use_json:
        with open(path, 'w', encoding='utf-8') as f:
            json.dump(obj, f, indent=2, ensure_ascii=False)
    else:
        joblib.dump(obj, path, compress=3)

    size_kb = path.stat().st_size / 1024
    print(f'  ✅  {path.name:<35}  {size_kb:>9.1f} KB  →  {path.resolve()}')


print('Helper functions defined')

Helper functions defined


---
## 4. Load & Prepare Data

Loads the processed CSVs and builds `X_train`, `X_test`, `y_train`, `y_test`  
along with the fitted `ord_enc`, `lbl_enc`, and `CLASSES` list.

In [9]:
def load_and_prepare(
    data_dir: Path,
    cat_features: list,
    num_features: list,
    target: str,
):
    """
    Load crop_train.csv and crop_test.csv from data_dir.
    Fit OrdinalEncoder on train categorical columns.
    Fit LabelEncoder on train target column.
    Return encoded numpy arrays and fitted encoders.

    Returns
    -------
    X_train, X_test  : np.ndarray  — encoded feature matrices
    y_train, y_test  : np.ndarray  — integer-encoded target vectors
    ord_enc          : fitted OrdinalEncoder
    lbl_enc          : fitted LabelEncoder
    classes          : list of str — class names in label-encoded order
    """
    train_df = pd.read_csv(data_dir / 'crop_train.csv')
    test_df  = pd.read_csv(data_dir / 'crop_test.csv')

    print(f'  Loaded  crop_train.csv  →  {train_df.shape}')
    print(f'  Loaded  crop_test.csv   →  {test_df.shape}')

    # ── Fit OrdinalEncoder on training categorical columns ──────────
    ord_enc = OrdinalEncoder(
        handle_unknown='use_encoded_value',
        unknown_value=-1,
    )
    ord_enc.fit(train_df[cat_features])

    # ── Fit LabelEncoder on training target ──────────────────────────
    lbl_enc = LabelEncoder()
    lbl_enc.fit(train_df[target])
    classes = lbl_enc.classes_.tolist()

    # ── Encode helper ────────────────────────────────────────────────
    def encode(df: pd.DataFrame) -> np.ndarray:
        X_cat = ord_enc.transform(df[cat_features])
        X_num = df[num_features].values.astype(float)
        return np.hstack([X_cat, X_num])

    X_train = encode(train_df)
    X_test  = encode(test_df)
    y_train = lbl_enc.transform(train_df[target])
    y_test  = lbl_enc.transform(test_df[target])

    return X_train, X_test, y_train, y_test, ord_enc, lbl_enc, classes


print('Preparing data ...')
X_train, X_test, y_train, y_test, ord_enc, lbl_enc, CLASSES = load_and_prepare(
    data_dir     = DATA_DIR,
    cat_features = CAT_FEATURES,
    num_features = NUM_FEATURES,
    target       = TARGET,
)

print()
print(f'  X_train  : {X_train.shape}  (dtype={X_train.dtype})')
print(f'  X_test   : {X_test.shape}  (dtype={X_test.dtype})')
print(f'  y_train  : {y_train.shape}  — {np.unique(y_train).size} unique classes')
print(f'  y_test   : {y_test.shape}  — {np.unique(y_test).size} unique classes')
print(f'  Classes  : {CLASSES}')
print()
print(f'  Class distribution (train):')
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'    {CLASSES[u]:<22}  {c:>7,}  ({c / len(y_train) * 100:.1f}%)')

Preparing data ...
  Loaded  crop_train.csv  →  (276261, 6)
  Loaded  crop_test.csv   →  (69066, 6)

  X_train  : (276261, 4)  (dtype=float64)
  X_test   : (69066, 4)  (dtype=float64)
  y_train  : (276261,)  — 9 unique classes
  y_test   : (69066,)  — 9 unique classes
  Classes  : ['Cash Crops', 'Cereals', 'Fiber Crops', 'Fruits', 'Oilseeds', 'Plantation Crops', 'Pulses', 'Spices', 'Vegetables']

  Class distribution (train):
    Cash Crops               11,977  (4.3%)
    Cereals                  72,911  (26.4%)
    Fiber Crops              11,028  (4.0%)
    Fruits                    3,366  (1.2%)
    Oilseeds                 52,756  (19.1%)
    Plantation Crops          5,315  (1.9%)
    Pulses                   70,403  (25.5%)
    Spices                   25,080  (9.1%)
    Vegetables               23,425  (8.5%)


---
## 5. Train XGBoost on Full Training Set

> XGBoost is chosen over Gradient Boosting (CV rank #1) due to identical  
> Top-3 accuracy (Δ = 0.0001) at 33× lower training time (105s vs 3439s).

In [10]:
print(f'Training {SELECTED_MODEL_NAME} on full training set ...')
print(f'  X_train : {X_train.shape}  |  Classes : {len(CLASSES)}')
print(f'  Params  : {XGBOOST_PARAMS}')
print()

xgb_model = XGBClassifier(**XGBOOST_PARAMS)

t0 = time.time()
xgb_model.fit(X_train, y_train)
train_time = time.time() - t0

fitted_model    = xgb_model
best_model_name = SELECTED_MODEL_NAME

print(f'  Fit complete in {train_time:.1f}s')
print(f'  n_classes seen : {xgb_model.n_classes_}')

Training XGBoost on full training set ...
  X_train : (276261, 4)  |  Classes : 9
  Params  : {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 7, 'subsample': 0.8, 'colsample_bytree': 0.8, 'eval_metric': 'mlogloss', 'use_label_encoder': False, 'n_jobs': -1, 'random_state': 42, 'verbosity': 0}

  Fit complete in 26.3s
  n_classes seen : 9


---
## 6. Test Set Metrics

In [11]:
print(f'Evaluating {best_model_name} on held-out test set ...')

TEST_METRICS = evaluate_on_test(fitted_model, X_test, y_test, k=TOP_K)

print(f"\n  {'Metric':<22} {'Score':>8}")
print(f"  {'─' * 32}")
for metric, value in TEST_METRICS.items():
    print(f"  {metric:<22} {value:>8.4f}")

Evaluating XGBoost on held-out test set ...

  Metric                    Score
  ────────────────────────────────
  test_accuracy            0.3806
  top3_accuracy            0.8600
  weighted_f1              0.3454
  macro_f1                 0.2533


---
## 7. Save Artifacts

In [12]:
print('Saving artifacts to:', MODELS_DIR.resolve())
print('─' * 80)

# ── 1. XGBoost model ────────────────────────────────────────────────
save_artifact(
    fitted_model,
    MODELS_DIR / 'crop_category_xgboost.pkl',
)

# ── 2. OrdinalEncoder ───────────────────────────────────────────────
save_artifact(
    ord_enc,
    MODELS_DIR / 'ordinal_encoder.pkl',
)

# ── 3. LabelEncoder ─────────────────────────────────────────────────
save_artifact(
    lbl_enc,
    MODELS_DIR / 'label_encoder.pkl',
)

# ── 4. Feature columns ──────────────────────────────────────────────
save_artifact(
    ALL_FEATURES,
    MODELS_DIR / 'feature_columns.json',
    use_json=True,
)

# ── 5. Class names ──────────────────────────────────────────────────
save_artifact(
    CLASSES,
    MODELS_DIR / 'crop_categories.json',
    use_json=True,
)

# ── 6. Metadata ─────────────────────────────────────────────────────
metadata = {
    'model_name'         : 'FarmIntel-Stage1-CropCategory',
    'algorithm'          : best_model_name,
    'training_date'      : datetime.datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    'number_of_features' : len(ALL_FEATURES),
    'feature_names'      : ALL_FEATURES,
    'number_of_classes'  : len(CLASSES),
    'class_names'        : CLASSES,
    f'top{TOP_K}_accuracy': TEST_METRICS[f'top{TOP_K}_accuracy'],
    'test_accuracy'      : TEST_METRICS['test_accuracy'],
    'weighted_f1'        : TEST_METRICS['weighted_f1'],
    'macro_f1'           : TEST_METRICS['macro_f1'],
    'sklearn_version'    : sklearn.__version__,
    'xgboost_version'    : xgb.__version__,
    'python_version'     : sys.version.split()[0],
    'xgboost_params'     : XGBOOST_PARAMS,
    'encoding'           : {
        'categorical_features' : CAT_FEATURES,
        'numerical_features'   : NUM_FEATURES,
        'encoder_type'         : 'OrdinalEncoder',
        'unknown_value'        : -1,
    },
    'stage'              : 1,
    'stage_description'  : 'Predicts Top-3 Crop Categories from State, District, Season, Year.',
    'next_stage'         : 'Stage 2 — within-category crop ranking by historical frequency.',
}

save_artifact(
    metadata,
    MODELS_DIR / 'metadata.json',
    use_json=True,
)

print('─' * 80)
print('All 6 artifacts saved successfully.')

Saving artifacts to: D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models
────────────────────────────────────────────────────────────────────────────────
  ✅  crop_category_xgboost.pkl               6212.1 KB  →  D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models\crop_category_xgboost.pkl
  ✅  ordinal_encoder.pkl                        4.8 KB  →  D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models\ordinal_encoder.pkl
  ✅  label_encoder.pkl                          0.4 KB  →  D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models\label_encoder.pkl
  ✅  feature_columns.json                       0.1 KB  →  D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models\feature_columns.json
  ✅  crop_categories.json                       0.1 KB  →  D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models\crop_categories.json
  ✅  metadata.json                              1.3 KB  →  D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models\metadata.json
──────────────────────────────────────────────

---
## 8. Verify Saved Artifacts

In [13]:
def verify_artifacts(models_dir: Path) -> None:
    """
    Reload every saved artifact and run sanity checks.
    Raises AssertionError if any file fails validation.
    """
    print('Running artifact verification ...\n')

    # Reload pkl files
    loaded_model = joblib.load(models_dir / 'crop_category_xgboost.pkl')
    loaded_ord   = joblib.load(models_dir / 'ordinal_encoder.pkl')
    loaded_lbl   = joblib.load(models_dir / 'label_encoder.pkl')

    # Reload JSON files
    with open(models_dir / 'feature_columns.json',  encoding='utf-8') as f:
        loaded_features = json.load(f)
    with open(models_dir / 'crop_categories.json',  encoding='utf-8') as f:
        loaded_classes  = json.load(f)
    with open(models_dir / 'metadata.json',          encoding='utf-8') as f:
        loaded_meta     = json.load(f)

    # Sanity checks
    sample       = X_test[:5]
    pred_orig    = fitted_model.predict(sample)
    pred_loaded  = loaded_model.predict(sample)

    assert np.array_equal(pred_orig, pred_loaded),              'Model predictions mismatch!'
    assert loaded_features == ALL_FEATURES,                     'Feature columns mismatch!'
    assert loaded_classes  == CLASSES,                          'Class names mismatch!'
    assert loaded_meta['algorithm'] == SELECTED_MODEL_NAME,     'Metadata algorithm mismatch!'
    assert len(loaded_ord.categories_) == len(CAT_FEATURES),    'OrdinalEncoder category count mismatch!'
    assert list(loaded_lbl.classes_)   == CLASSES,              'LabelEncoder classes mismatch!'

    checks = [
        ('crop_category_xgboost.pkl', 'Predictions match original model'),
        ('ordinal_encoder.pkl',       f'{len(loaded_ord.categories_)} categorical columns encoded'),
        ('label_encoder.pkl',         f'{len(loaded_lbl.classes_)} classes encoded'),
        ('feature_columns.json',      f'{len(loaded_features)} features: {loaded_features}'),
        ('crop_categories.json',      f'{len(loaded_classes)} classes: {loaded_classes}'),
        ('metadata.json',             f'algorithm={loaded_meta["algorithm"]}  top3={loaded_meta.get("top3_accuracy", "N/A")}'),
    ]
    for filename, note in checks:
        print(f'   {filename:<35}  {note}')

    print('\n All verification checks passed.')


verify_artifacts(MODELS_DIR)

Running artifact verification ...

   crop_category_xgboost.pkl            Predictions match original model
   ordinal_encoder.pkl                  3 categorical columns encoded
   label_encoder.pkl                    9 classes encoded
   feature_columns.json                 4 features: ['State', 'District', 'Season', 'Year']
   crop_categories.json                 9 classes: ['Cash Crops', 'Cereals', 'Fiber Crops', 'Fruits', 'Oilseeds', 'Plantation Crops', 'Pulses', 'Spices', 'Vegetables']
   metadata.json                        algorithm=XGBoost  top3=0.860003

 All verification checks passed.


---
## 9. Directory Tree

In [14]:
def print_directory_tree(root: Path, prefix: str = '') -> None:
    """Recursively print a directory tree with file sizes."""
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for i, entry in enumerate(entries):
        connector = '└── ' if i == len(entries) - 1 else '├── '
        if entry.is_dir():
            print(f'{prefix}{connector}{entry.name}/')
            extension = '    ' if i == len(entries) - 1 else '│   '
            print_directory_tree(entry, prefix + extension)
        else:
            size_kb = entry.stat().st_size / 1024
            unit    = 'MB' if size_kb >= 1024 else 'KB'
            size    = size_kb / 1024 if size_kb >= 1024 else size_kb
            print(f'{prefix}{connector}{entry.name}  ({size:.1f} {unit})')


print(f'{MODELS_DIR.resolve()}/')
print_directory_tree(MODELS_DIR)

total_bytes = sum(f.stat().st_size for f in MODELS_DIR.rglob('*') if f.is_file())
total_mb    = total_bytes / (1024 ** 2)
n_files     = len([f for f in MODELS_DIR.iterdir() if f.is_file()])
print(f'\nTotal: {n_files} files  |  {total_mb:.2f} MB')

D:\PROJECTS\AGRICULTURE\FarmIntel\ml-models-v2\models/
├── crop_categories.json  (0.1 KB)
├── crop_category_xgboost.pkl  (6.1 MB)
├── feature_columns.json  (0.1 KB)
├── label_encoder.pkl  (0.4 KB)
├── metadata.json  (1.3 KB)
└── ordinal_encoder.pkl  (4.8 KB)

Total: 6 files  |  6.07 MB


---
## 10. Metadata Preview

In [15]:
with open(MODELS_DIR / 'metadata.json', encoding='utf-8') as f:
    saved_meta = json.load(f)

print(json.dumps(saved_meta, indent=2))

{
  "model_name": "FarmIntel-Stage1-CropCategory",
  "algorithm": "XGBoost",
  "training_date": "2026-07-06T17:26:57Z",
  "number_of_features": 4,
  "feature_names": [
    "State",
    "District",
    "Season",
    "Year"
  ],
  "number_of_classes": 9,
  "class_names": [
    "Cash Crops",
    "Cereals",
    "Fiber Crops",
    "Fruits",
    "Oilseeds",
    "Plantation Crops",
    "Pulses",
    "Spices",
    "Vegetables"
  ],
  "top3_accuracy": 0.860003,
  "test_accuracy": 0.380607,
  "weighted_f1": 0.345447,
  "macro_f1": 0.253281,
  "sklearn_version": "1.7.2",
  "xgboost_version": "3.2.0",
  "python_version": "3.10.10",
  "xgboost_params": {
    "n_estimators": 300,
    "learning_rate": 0.05,
    "max_depth": 7,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "eval_metric": "mlogloss",
    "use_label_encoder": false,
    "n_jobs": -1,
    "random_state": 42,
    "verbosity": 0
  },
  "encoding": {
    "categorical_features": [
      "State",
      "District",
      "Season"
    

---
## 11. Inference Smoke Test

End-to-end: raw string input → encode → predict → decode Top-3 categories.

In [16]:
def predict_top_k_categories(
    state: str,
    district: str,
    season: str,
    year: int,
    k: int = TOP_K,
    model_dir: Path = MODELS_DIR,
) -> list:
    """
    End-to-end inference — loads artifacts from disk.

    Returns
    -------
    list of dicts: [{'rank': 1, 'category': '...', 'probability': 0.xx}, ...]
    """
    _model   = joblib.load(model_dir / 'crop_category_xgboost.pkl')
    _ord_enc = joblib.load(model_dir / 'ordinal_encoder.pkl')
    _lbl_enc = joblib.load(model_dir / 'label_encoder.pkl')

    with open(model_dir / 'feature_columns.json', encoding='utf-8') as f:
        _features = json.load(f)

    _cat_cols = [c for c in _features if c != 'Year']
    _num_cols = ['Year']

    raw_df    = pd.DataFrame([{'State': state, 'District': district,
                                'Season': season, 'Year': year}])[_features]
    X_encoded = np.hstack([
        _ord_enc.transform(raw_df[_cat_cols]),
        raw_df[_num_cols].values.astype(float),
    ])

    proba       = _model.predict_proba(X_encoded)[0]
    top_k_idx   = np.argsort(proba)[::-1][:k]
    top_k_names = _lbl_enc.inverse_transform(top_k_idx)

    return [
        {'rank': r + 1, 'category': cat, 'probability': round(float(proba[idx]), 4)}
        for r, (cat, idx) in enumerate(zip(top_k_names, top_k_idx))
    ]


# ── Test cases ───────────────────────────────────────────────────────
test_inputs = [
    ('Uttar Pradesh', 'Faizabad',  'Rabi',       2005),
    ('Punjab',        'Ludhiana',  'Kharif',      2015),
    ('Kerala',        'Wayanad',   'Whole Year',  2010),
    ('Maharashtra',   'Pune',      'Kharif',      2018),
    ('Rajasthan',     'Jaipur',    'Rabi',        2012),
]

print(f'Inference Smoke Test — Top-{TOP_K} Crop Category Predictions')
print('=' * 65)
for state, district, season, year in test_inputs:
    preds = predict_top_k_categories(state, district, season, year)
    print(f'\n  Input : {state} | {district} | {season} | {year}')
    for p in preds:
        bar = '█' * int(p['probability'] * 30)
        print(f"  #{p['rank']}  {p['category']:<22}  {p['probability']:.4f}  {bar}")

print('\n Smoke test complete — inference pipeline working correctly.')

Inference Smoke Test — Top-3 Crop Category Predictions

  Input : Uttar Pradesh | Faizabad | Rabi | 2005
  #1  Cereals                 0.3642  ██████████
  #2  Oilseeds                0.2319  ██████
  #3  Pulses                  0.1836  █████

  Input : Punjab | Ludhiana | Kharif | 2015
  #1  Cereals                 0.4268  ████████████
  #2  Pulses                  0.2994  ████████
  #3  Oilseeds                0.1487  ████

  Input : Kerala | Wayanad | Whole Year | 2010
  #1  Spices                  0.3356  ██████████
  #2  Plantation Crops        0.2662  ███████
  #3  Vegetables              0.1706  █████

  Input : Maharashtra | Pune | Kharif | 2018
  #1  Cereals                 0.3469  ██████████
  #2  Oilseeds                0.3174  █████████
  #3  Pulses                  0.2774  ████████

  Input : Rajasthan | Jaipur | Rabi | 2012
  #1  Pulses                  0.3553  ██████████
  #2  Oilseeds                0.3542  ██████████
  #3  Cereals                 0.2710  ████████

 Smo

---
## 12. Final Summary

In [18]:
print('=' * 62)
print('  FARMINTEL STAGE 1 — ARTIFACT SAVE COMPLETE')
print('=' * 62)
print(f'  Algorithm          : {SELECTED_MODEL_NAME}')
print(f'  Training samples   : {len(y_train):,}')
print(f'  Test  samples      : {len(y_test):,}')
print(f'  Classes ({len(CLASSES)})        : {CLASSES}')
print(f'  Features           : {ALL_FEATURES}')
print()
print('  ── Test Set Metrics ──')
for k, v in TEST_METRICS.items():
    print(f'  {k:<22}: {v:.4f}')
print()
print('  ── Saved Files ──')
for f in sorted(MODELS_DIR.iterdir()):
    if f.is_file():
        size_kb = f.stat().st_size / 1024
        unit    = 'MB' if size_kb >= 1024 else 'KB'
        size    = size_kb / 1024 if size_kb >= 1024 else size_kb
        print(f'  • {f.name:<35}  {size:>7.1f} {unit}')
print()
print('    Next: Stage 2 — within-category crop ranking')
print('          using historical frequencies.')
print('=' * 62)

  FARMINTEL STAGE 1 — ARTIFACT SAVE COMPLETE
  Algorithm          : XGBoost
  Training samples   : 276,261
  Test  samples      : 69,066
  Classes (9)        : ['Cash Crops', 'Cereals', 'Fiber Crops', 'Fruits', 'Oilseeds', 'Plantation Crops', 'Pulses', 'Spices', 'Vegetables']
  Features           : ['State', 'District', 'Season', 'Year']

  ── Test Set Metrics ──
  test_accuracy         : 0.3806
  top3_accuracy         : 0.8600
  weighted_f1           : 0.3454
  macro_f1              : 0.2533

  ── Saved Files ──
  • crop_categories.json                     0.1 KB
  • crop_category_xgboost.pkl                6.1 MB
  • feature_columns.json                     0.1 KB
  • label_encoder.pkl                        0.4 KB
  • metadata.json                            1.3 KB
  • ordinal_encoder.pkl                      4.8 KB

    Next: Stage 2 — within-category crop ranking
          using historical frequencies.
